In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_PATH       = '/content/drive/MyDrive/BrainMahima'
CHECKPOINT_PATH  = f'{DRIVE_PATH}/best_model.h5'
CROP_TRAIN_PATH  = f'{DRIVE_PATH}/Crop-Brain-MRI'
CROP_TEST_PATH   = f'{DRIVE_PATH}/Test-Brain-MRI'

os.makedirs(DRIVE_PATH, exist_ok=True)
print("✓ Drive Ready")

Mounted at /content/drive
✓ Drive Ready


In [ ]:
!pip install kaggle imutils tqdm -q

import os, glob, random, shutil
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import cv2, imutils
import tensorflow as tf

from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import mixed_precision
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

mixed_precision.set_global_policy('mixed_float16')

print("✓ TensorFlow:", tf.__version__)

✓ TensorFlow: 2.19.0


In [ ]:
from google.colab import files

files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d sartajbhuvaji/brain-tumor-classification-mri -p /content/ --force

!unzip -q -o "/content/*.zip" -d /content/data

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/sartajbhuvaji/brain-tumor-classification-mri
License(s): MIT
100% 86.8M/86.8M [00:06<00:00, 14.8MB/s]



In [ ]:
train_dir = "/content/data/Training/"
test_dir  = "/content/data/Testing/"

classes = sorted(os.listdir(train_dir))
print(classes)

['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']


In [ ]:
def crop_image(image):
    try:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (5,5), 0)

        thresh = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)[1]
        thresh = cv2.erode(thresh, None, iterations=2)
        thresh = cv2.dilate(thresh, None, iterations=2)

        contours = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contours = imutils.grab_contours(contours)

        if not contours:
            return image   # ✅ fallback (IMPORTANT)

        c = max(contours, key=cv2.contourArea)

        extLeft  = tuple(c[c[:, :, 0].argmin()][0])
        extRight = tuple(c[c[:, :, 0].argmax()][0])
        extTop   = tuple(c[c[:, :, 1].argmin()][0])
        extBot   = tuple(c[c[:, :, 1].argmax()][0])

        new_image = image[extTop[1]:extBot[1], extLeft[0]:extRight[0]]

        if new_image.size == 0:
            return image  # fallback

        return new_image
    except:
        return image

In [ ]:
def process_data(src, dst):
    os.makedirs(dst, exist_ok=True)

    for cls in classes:
        src_cls = os.path.join(src, cls)
        dst_cls = os.path.join(dst, cls)
        os.makedirs(dst_cls, exist_ok=True)

        for i, file in enumerate(os.listdir(src_cls)):
            img = cv2.imread(os.path.join(src_cls, file))
            if img is None:
                continue

            cropped = crop_image(img)
            resized = cv2.resize(cropped, (240,240))
            cv2.imwrite(os.path.join(dst_cls, f"{i}.jpg"), resized)

process_data(train_dir, CROP_TRAIN_PATH)
process_data(test_dir, CROP_TEST_PATH)

print("✓ Cropping Done")

✓ Cropping Done


In [ ]:
IMG_SIZE = (240,240)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_dataset(path, shuffle=True):
    return tf.keras.utils.image_dataset_from_directory(
        path,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        label_mode='categorical'
    )

full_ds = load_dataset(CROP_TRAIN_PATH)

val_size = int(0.2 * len(full_ds))
val_ds   = full_ds.take(val_size)
train_ds = full_ds.skip(val_size)

# Strong augmentation
aug = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

train_ds = train_ds.map(lambda x,y: (aug(x, training=True), y))
train_ds = train_ds.map(lambda x,y: (preprocess_input(x), y)).prefetch(AUTOTUNE)
val_ds   = val_ds.map(lambda x,y: (preprocess_input(x), y)).prefetch(AUTOTUNE)

test_ds  = load_dataset(CROP_TEST_PATH, shuffle=False)
test_ds  = test_ds.map(lambda x,y: (preprocess_input(x), y)).prefetch(AUTOTUNE)

print("✓ Dataset Ready")

Found 2870 files belonging to 4 classes.
Found 394 files belonging to 4 classes.
✓ Dataset Ready


In [ ]:
labels = []
for _, y in train_ds.unbatch():
    labels.append(np.argmax(y.numpy()))

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(class_weights))
print(class_weights)

{0: np.float64(0.8508902077151336), 1: np.float64(0.8650075414781297), 2: np.float64(1.8034591194968554), 3: np.float64(0.8974960876369327)}


In [ ]:
base = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(240,240,3))
base.trainable = False

x = base.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)

output = Dense(4, activation='softmax', dtype='float32')(x)

model = Model(inputs=base.input, outputs=output)

model.compile(
    optimizer=Adam(1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

model.summary()

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 240, 240,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 240, 240,  │          0 │ input_layer_1[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 240, 240,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 240, 240,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 241, 241,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 120, 120,  │      1,080 │ stem_conv_pad[0]… │
│                     │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 120, 120,  │        160 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 120, 120,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 120, 120,  │        360 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 120, 120,  │        160 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 120, 120,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 40)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 40)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 40)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 10)  │        410 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 40)  │        440 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 120, 120,  │          0 │ block1a_activati… │
│ (Multiply)          │ 40)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 120, 120,  │        960 │ block1a_se_excit

 Total params: 11,216,563 (42.79 MB)

 Trainable params: 429,956 (1.64 MB)

 Non-trainable params: 10,786,607 (41.15 MB)

In [ ]:
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ModelCheckpoint(CHECKPOINT_PATH, save_best_only=True),
    ReduceLROnPlateau(patience=3)
]

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks
)

Epoch 1/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4664 - loss: 1.5653   

72/72 ━━━━━━━━━━━━━━━━━━━━ 253s 2s/step - accuracy: 0.5514 - loss: 1.3371 - val_accuracy: 0.6962 - val_loss: 0.9362 - learning_rate: 0.0010
Epoch 2/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 682ms/step - accuracy: 0.6315 - loss: 1.0415

72/72 ━━━━━━━━━━━━━━━━━━━━ 57s 749ms/step - accuracy: 0.6486 - loss: 1.0169 - val_accuracy: 0.7899 - val_loss: 0.8572 - learning_rate: 0.0010
Epoch 3/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 689ms/step - accuracy: 0.6640 - loss: 0.9827

72/72 ━━━━━━━━━━━━━━━━━━━━ 57s 754ms/step - accuracy: 0.6857 - loss: 0.9391 - val_accuracy: 0.7847 - val_loss: 0.8260 - learning_rate: 0.0010
Epoch 4/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 685ms/step - accuracy: 0.7187 - loss: 0.8605

72/72 ━━━━━━━━━━━━━━━━━━━━ 56s 746ms/step - accuracy: 0.7254 - loss: 0.8625 - val_accuracy: 0.8021 - val_loss: 0.7952 - learning_rate: 0.0010
Epoch 5/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 691ms/step - accuracy: 0.7313 - loss: 0.8323

...

72/72 ━━━━━━━━━━━━━━━━━━━━ 56s 749ms/step - accuracy: 0.7694 - loss: 0.7805 - val_accuracy: 0.8212 - val_loss: 0.6988 - learning_rate: 0.0010
Epoch 10/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 679ms/step - accuracy: 0.7639 - loss: 0.7762

72/72 ━━━━━━━━━━━━━━━━━━━━ 56s 750ms/step - accuracy: 0.7807 - loss: 0.7584 - val_accuracy: 0.8333 - val_loss: 0.6883 - learning_rate: 0.0010
Epoch 11/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 686ms/step - accuracy: 0.7711 - loss: 0.7498

72/72 ━━━━━━━━━━━━━━━━━━━━ 57s 762ms/step - accuracy: 0.7812 - loss: 0.7420 - val_accuracy: 0.8385 - val_loss: 0.6746 - learning_rate: 0.0010
Epoch 12/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 679ms/step - accuracy: 0.7827 - loss: 0.7487

72/72 ━━━━━━━━━━━━━━━━━━━━ 81s 752ms/step - accuracy: 0.7929 - loss: 0.7387 - val_accuracy: 0.8438 - val_loss: 0.6695 - learning_rate: 0.0010
Epoch 13/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 710ms/step - accuracy: 0.8039 - loss: 0.7364

72/72 ━━━━━━━━━━━━━━━━━━━━ 59s 785ms/step - accuracy: 0.8056 - loss: 0.7325 - val_accuracy: 0.8542 - val_loss: 0.6489 - learning_rate: 0.0010
Epoch 14/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 55s 730ms/step - accuracy: 0.8091 - loss: 0.7117 - val_accuracy: 0.8524 - val_loss: 0.6520 - learning_rate: 0.0010
Epoch 15/15
72/72 ━━━━━━━━━━━━━━━━━━━━ 53s 703ms/step - accuracy: 0.8091 - loss: 0.7050 - val_accuracy: 0.8316 - val_loss: 0.6542 - learning_rate: 0.0010


In [ ]:
base.trainable = True

for layer in base.layers[:-60]:
    layer.trainable = False

model.compile(
    optimizer=Adam(3e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    class_weight=class_weights,
    callbacks=callbacks
)

Epoch 1/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7017 - loss: 0.9006   

72/72 ━━━━━━━━━━━━━━━━━━━━ 254s 2s/step - accuracy: 0.7201 - loss: 0.8605 - val_accuracy: 0.8611 - val_loss: 0.6332 - learning_rate: 3.0000e-05
Epoch 2/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 56s 745ms/step - accuracy: 0.7707 - loss: 0.7685 - val_accuracy: 0.8403 - val_loss: 0.6679 - learning_rate: 3.0000e-05
Epoch 3/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 679ms/step - accuracy: 0.7867 - loss: 0.7516

72/72 ━━━━━━━━━━━━━━━━━━━━ 56s 758ms/step - accuracy: 0.8012 - loss: 0.7385 - val_accuracy: 0.8681 - val_loss: 0.6277 - learning_rate: 3.0000e-05
Epoch 4/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 56s 745ms/step - accuracy: 0.8191 - loss: 0.7090 - val_accuracy: 0.8715 - val_loss: 0.6311 - learning_rate: 3.0000e-05
Epoch 5/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 53s 715ms/step - accuracy: 0.8361 - loss: 0.6718 - val_accuracy: 0.8646 - val_loss: 0.6409 - learning_rate: 3.0000e-05
Epoch 6/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 686ms/step - accuracy: 0.8061 - loss: 0.6943

72/72 ━━━━━━━━━━━━━━━━━━━━ 58s 764ms/step - accuracy: 0.8282 - loss: 0.6800 - val_accuracy: 0.8733 - val_loss: 0.6213 - learning_rate: 3.0000e-05
Epoch 7/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 704ms/step - accuracy: 0.8399 - loss: 0.6644

72/72 ━━━━━━━━━━━━━━━━━━━━ 60s 795ms/step - accuracy: 0.8391 - loss: 0.6672 - val_accuracy: 0.8854 - val_loss: 0.6094 - learning_rate: 3.0000e-05
Epoch 8/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 56s 754ms/step - accuracy: 0.8666 - loss: 0.6362 - val_accuracy: 0.8854 - val_loss: 0.6113 - learning_rate: 3.0000e-05
Epoch 9/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 685ms/step - accuracy: 0.8647 - loss: 0.6384

...

72/72 ━━━━━━━━━━━━━━━━━━━━ 57s 757ms/step - accuracy: 0.8923 - loss: 0.6020 - val_accuracy: 0.9097 - val_loss: 0.5721 - learning_rate: 3.0000e-05
Epoch 13/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 78s 707ms/step - accuracy: 0.8971 - loss: 0.5882 - val_accuracy: 0.8906 - val_loss: 0.5733 - learning_rate: 3.0000e-05
Epoch 14/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 679ms/step - accuracy: 0.8990 - loss: 0.5822

72/72 ━━━━━━━━━━━━━━━━━━━━ 86s 767ms/step - accuracy: 0.9028 - loss: 0.5822 - val_accuracy: 0.9028 - val_loss: 0.5564 - learning_rate: 3.0000e-05
Epoch 15/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 56s 751ms/step - accuracy: 0.9010 - loss: 0.5876 - val_accuracy: 0.9062 - val_loss: 0.5596 - learning_rate: 3.0000e-05
Epoch 16/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 53s 713ms/step - accuracy: 0.8941 - loss: 0.5912 - val_accuracy: 0.8958 - val_loss: 0.5639 - learning_rate: 3.0000e-05
Epoch 17/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 53s 705ms/step - accuracy: 0.9058 - loss: 0.5771 - val_accuracy: 0.9010 - val_loss: 0.5580 - learning_rate: 3.0000e-05
Epoch 18/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 695ms/step - accuracy: 0.9163 - loss: 0.5561

72/72 ━━━━━━━━━━━━━━━━━━━━ 60s 804ms/step - accuracy: 0.9180 - loss: 0.5558 - val_accuracy: 0.9062 - val_loss: 0.5494 - learning_rate: 3.0000e-06
Epoch 19/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 709ms/step - accuracy: 0.9096 - loss: 0.5746

72/72 ━━━━━━━━━━━━━━━━━━━━ 60s 803ms/step - accuracy: 0.9154 - loss: 0.5633 - val_accuracy: 0.9028 - val_loss: 0.5490 - learning_rate: 3.0000e-06
Epoch 20/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 75s 709ms/step - accuracy: 0.9207 - loss: 0.5582 - val_accuracy: 0.9028 - val_loss: 0.5518 - learning_rate: 3.0000e-06
Epoch 21/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 687ms/step - accuracy: 0.8986 - loss: 0.5742

72/72 ━━━━━━━━━━━━━━━━━━━━ 57s 761ms/step - accuracy: 0.9085 - loss: 0.5660 - val_accuracy: 0.9167 - val_loss: 0.5414 - learning_rate: 3.0000e-06
Epoch 22/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 56s 736ms/step - accuracy: 0.9119 - loss: 0.5611 - val_accuracy: 0.9097 - val_loss: 0.5451 - learning_rate: 3.0000e-06
Epoch 23/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 54s 720ms/step - accuracy: 0.9185 - loss: 0.5672 - val_accuracy: 0.9028 - val_loss: 0.5575 - learning_rate: 3.0000e-06
Epoch 24/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 82s 718ms/step - accuracy: 0.9211 - loss: 0.5630 - val_accuracy: 0.9097 - val_loss: 0.5448 - learning_rate: 3.0000e-06
Epoch 25/25
72/72 ━━━━━━━━━━━━━━━━━━━━ 54s 723ms/step - accuracy: 0.9172 - loss: 0.5503 - val_accuracy: 0.9097 - val_loss: 0.5462 - learning_rate: 3.0000e-07


In [ ]:
model.load_weights(CHECKPOINT_PATH)

loss, acc = model.evaluate(test_ds)
print(f"Test Accuracy: {acc*100:.2f}%")

13/13 ━━━━━━━━━━━━━━━━━━━━ 54s 4s/step - accuracy: 0.8020 - loss: 0.7600
Test Accuracy: 80.20%


In [ ]:
y_pred = np.argmax(model.predict(test_ds), axis=1)
y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds])

print(classification_report(y_true, y_pred))

13/13 ━━━━━━━━━━━━━━━━━━━━ 59s 2s/step
              precision    recall  f1-score   support

           0       0.84      0.52      0.64       100
           1       0.74      0.86      0.80       115
           2       0.82      0.98      0.89       105
           3       0.85      0.84      0.84        74

    accuracy                           0.80       394
   macro avg       0.81      0.80      0.79       394
weighted avg       0.81      0.80      0.79       394



In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

print(os.listdir('/content/drive/MyDrive'))
print(os.listdir('/content/drive/MyDrive/BrainMahima'))

['Colab Notebooks', 'Maths', 'L.Mahima.pdf', 'Python ', 'Mahima ', 'New', 'Gd (1).gdoc', 'Gd.gdoc', 'Untitled presentation (6).gslides', 'Untitled presentation (5).gslides', 'Untitled presentation (4).gslides', 'Untitled presentation (3).gslides', 'Untitled presentation (2).gslides', 'Untitled presentation (1).gslides', 'Untitled presentation.gslides', 'Part B important questions .pdf', 'Part B important questions .gdoc', 'bank.gdoc', 'Cache.gdoc', 'done.gdoc', 'Hello .gdoc', 'For me.gdoc', 'Untitled document (2).gdoc', 'IMG-20231123-WA0020.jpg', 'Javafx.gdoc', 'MAHIMA L profile entry.docx', 'Untitled document (1).gdoc', 'import javafx.gdoc', 'Untitled document.gdoc', 'Party Invite.gform', 'Untitled form (1).gform', 'Uipath certificates', 'Untitled form.gform', 'Bannar BookCafe', 'Screenshot_2024-10-05-18-15-29-79.jpg', 'Attendance NM', '37_MAHIMA L.pdf', 'DM Example.gslides', 'MahimaCertificates', 'CD Ex 7', 'EX NO1.pdf', 'portfolio.css', 'style.css', '713 - Web Technologies', 'Co  PO

In [6]:
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.models import Model

base = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(240,240,3))

x = base.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)

output = Dense(4, activation='softmax')(x)

model = Model(inputs=base.input, outputs=output)

In [7]:
model.load_weights('/content/drive/MyDrive/BrainMahima/best_model.h5')

print("✅ Model loaded successfully")

✅ Model loaded successfully


In [9]:
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input

IMG_SIZE = (240,240)
BATCH_SIZE = 32

test_ds = tf.keras.utils.image_dataset_from_directory(
    '/content/drive/MyDrive/BrainMahima/Test-Brain-MRI',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False  # IMPORTANT
)

# Apply preprocessing
test_ds = test_ds.map(lambda x, y: (preprocess_input(x), y))

Found 394 files belonging to 4 classes.


In [11]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

y_pred = np.argmax(model.predict(test_ds), axis=1)

y_true = np.concatenate([
    y.numpy() for _, y in test_ds
])

cm = confusion_matrix(y_true, y_pred)

print("CONFUSION MATRIX")
print(cm)

print("\nCLASSIFICATION REPORT")
print(classification_report(y_true, y_pred))

13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 188ms/step
CONFUSION MATRIX
[[ 52  29  16   3]
 [  4  99   4   8]
 [  0   2 103   0]
 [  6   3   3  62]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.84      0.52      0.64       100
           1       0.74      0.86      0.80       115
           2       0.82      0.98      0.89       105
           3       0.85      0.84      0.84        74

    accuracy                           0.80       394
   macro avg       0.81      0.80      0.79       394
weighted avg       0.81      0.80      0.79       394

